# Review Base Rag

In [1]:
# Set up the OpenAI client
from dotenv import  load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
# Load date and build the search engine
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

In [3]:
# Create the assistant
assistant = RAGBase(
    index=index,
    llm_client=openai_client)


In [4]:
assistant.rag("How do I run Ollama locally?")

'Run Ollama locally by:\n\n1. Installing it from https://ollama.com/download for your OS:\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Starting a model in your terminal:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the model and open a local chat interface.\n\n3. Optionally test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you want to use it from Python, install the client with:\n\n```bash\npip install ollama\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t know.'

# Agentic Rag

In [6]:
message = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message
)

response.output_text

'Yes, likely — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Please send me one of these:\n- the course name or link\n- the school/platform it’s on\n- whether you mean an online course, university class, or training program\n\nThen I can help you check:\n- if late enrollment is allowed\n- deadlines\n- prerequisites\n- who to contact to join'

In [7]:
import json

print(response.model_dump_json(indent=2))

{
  "id": "resp_046e22bbd90ffdbc006a398ed9ce5881a294a7d8893535afd1",
  "created_at": 1782157017.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_046e22bbd90ffdbc006a398eda6d1881a29c6947d724715563",
      "content": [
        {
          "annotations": [],
          "text": "Yes, likely — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Please send me one of these:\n- the course name or link\n- the school/platform it’s on\n- whether you mean an online course, university class, or training program\n\nThen I can help you check:\n- if late enrollment is allowed\n- deadlines\n- prerequisites\n- who to contact to join",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "typ

In [8]:
def search(query):
    boost_dic = {"question": 3.0, "section": 0.5}
    filter_dic = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dic,
        filter_dict=filter_dic
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message,
    tools = [search_tool]
)   

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enroll registration eligibility"}', call_id='call_omUezvQqpfKygWHxglw90Vyq', name='search', type='function_call', id='fc_00202b9175a08cec006a398ede41e481a180d5e481c22d147a', namespace=None, status='completed')]

In [11]:
response.output_text

''

In [12]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_00202b9175a08cec006a398edd868c81a1a7a43040b49d066d",
  "created_at": 1782157021.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "arguments": "{\"query\":\"join course discovered course can I join late enroll registration eligibility\"}",
      "call_id": "call_omUezvQqpfKygWHxglw90Vyq",
      "name": "search",
      "type": "function_call",
      "id": "fc_00202b9175a08cec006a398ede41e481a180d5e481c22d147a",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "search",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Search query text to look up in the course FAQ."
          }
        },
        "required": [
          

In [13]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
message.extend(response.output)
message.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
})

In [15]:
response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message,
    tools = [search_tool]
)
response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still open.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 34)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Agentic Loop

## A developer prompt


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## A function-call helper

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        results = search(**args)

    results_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": results_json
    }



In [20]:
question = "I just discovered the course. Can I join it?"

message = [{"role": "developer", "content": instructions},
           {"role": "user", "content": question}]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message,
    tools=[search_tool]
)

message.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        message.append(call_output)
        has_function_calls = True

    elif item.type  == "message":
        print("ASSISTANT:")
        print(item.content[0].text)


function_call: search {"query":"join course discovered course can I join enrollment late registration"}
function_call: search {"query":"course discovered can I still join FAQ enrollment"}
function_call: search {"query":"late join course registration enrollment discovered the course"}


## The full agent loop

In [21]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=message,
        tools=[search_tool]
    )


    message.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            message.append(call_output)
            has_function_calls = True

        elif item.type  == "message":
            print("ASSISTANT:")
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning anytime using the course materials.
- If you want a certificate, you need to submit your project while submissions are still open.
- Certificates are only available when you finish with a live cohort, not in self-paced mode.

If you want, I can also help you find the course docs, homework workflow, or certificate requirements. Is there anything else you’d like to explore?


## Wrapping it in a function


In [22]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            print("ITEM TYPE:", item.type)
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            
            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)
            
        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [23]:
agent_loop(instructions, "How do I run Olama locally?")


iteration #1...


ITEM TYPE: function_call
function_call: search {"query":"Olama locally run install local server Ollama"}
ITEM TYPE: function_call
function_call: search {"query":"run Ollama locally course FAQ"}
ITEM TYPE: function_call
function_call: search {"query":"Ollama local setup model run terminal"}
iteration #2...
ITEM TYPE: message
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat interface in your terminal.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the local Ollama server.

4. **Use it from Python**
   ```bash
   pip install o

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat interface in your terminal.\n\n3. **Check that the server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the local Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n``

## Encouraging multiple searches

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

You must use the search function before answering.
You must perform at least two sequential searches.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"join course discovered can I join enrollment late FAQ"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"late join project submission accepting submissions certificate live cohort self-paced FAQ"}
iteration #3...
ITEM TYPE: message
ASSISTANT:
Yes, you can still join the course.

According to the course FAQ, even if you just discovered it, you can start now. If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help explain how to get started with the course workflow. Are there other areas you want to explore?


'Yes, you can still join the course.\n\nAccording to the course FAQ, even if you just discovered it, you can start now. If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.\n\nIf you want, I can also help explain how to get started with the course workflow. Are there other areas you want to explore?'

## Restricting off-topic questions


In [25]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit chess opening"}
iteration #4...
ITEM TYPE: message
ASSISTANT:
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to draw Black’s d-pawn away from the center and gain control of the board. It’s one of the oldest and most classic chess openings.

A common response from Black is the **Queen’s Gambit Declined** if Black doesn’t take the pawn, or **Queen’s Gambit Accepted** if Black does take it.

If you want, I can also explain:
- the main ideas behind it,
- the difference between Accepted and Declined,
- or show a few beginner-friendly lines.

Is there another area you want to explore?


'A **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to draw Black’s d-pawn away from the center and gain control of the board. It’s one of the oldest and most classic chess openings.\n\nA common response from Black is the **Queen’s Gambit Declined** if Black doesn’t take the pawn, or **Queen’s Gambit Accepted** if Black does take it.\n\nIf you want, I can also explain:\n- the main ideas behind it,\n- the difference between Accepted and Declined,\n- or show a few beginner-friendly lines.\n\nIs there another area you want to explore?'

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit queen's gambit chess opening course faq"}
iteration #4...
ITEM TYPE: message
ASSISTANT:
I couldn’t find a course FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question.

If you meant **“Queen’s Gambit”** in chess, that’s a chess opening — but I don’t have a course FAQ entry to explain it here.

If you want, I can help with other course-related questions or logistics.


'I couldn’t find a course FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question.\n\nIf you meant **“Queen’s Gambit”** in chess, that’s a chess opening — but I don’t have a course FAQ entry to explain it here.\n\nIf you want, I can help with other course-related questions or logistics.'

# ToyAIKit

In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [29]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [32]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [33]:
result.cost

CostInfo(input_cost=Decimal('0.00280575'), output_cost=Decimal('0.001413'), total_cost=Decimal('0.00421875'))

In [34]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install FAQ"}', call_id='call_PR4jOj65

In [35]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [36]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how does this framework work', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"framework work how does this framework 